# Spatial error analysis and block bootstrap

Compare saved 2025 predictions, historical demand groups, spatial outliers, and demand contrast. Daily blocks preserve temporal ordering within blocks. The zone investigations are exploratory because 2025 outcomes motivated them.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import pandas as pd
import numpy as np
from arch.bootstrap import MovingBlockBootstrap
from data_tools import *
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt


In [ ]:
pred = pd.read_parquet(str(project_root / "data/predictions_2025.parquet"))
df_2025 = pd.read_parquet(str(project_root / "data/yellow_taxi_2025_features.parquet"))
df_2025_spatial = pd.read_parquet(str(project_root / "data/yellow_taxi_2025_features_neighbor_mean.parquet"))
df_2024 = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features.parquet"))
df_2024_spatial = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features_neighbor_mean.parquet"))

pred = pred.drop(columns = ["pred_shock"])

pred["day"] = df_2025["tpep_pickup_datetime"].dt.day
pred["month"] = df_2025["tpep_pickup_datetime"].dt.month
pred["hour"] = df_2025["tpep_pickup_datetime"].dt.hour
pred["zone"] = df_2025["PULocationID"]

pred["real"] = df_2025["trip_count"]


In [ ]:
pred["loss_diff"] = (pred["pred_basic"]-pred["real"])**2 - (pred["pred_mean"] - pred["real"])**2

pred["loss_mean"] = pred.groupby(["day", "month"])["loss_diff"].transform("mean")


## Moving-block bootstrap


In [ ]:
def ci(n):
    daily_loss = (pred.assign(date = df_2025["tpep_pickup_datetime"].dt.normalize())
                  .groupby("date")["loss_diff"].mean().sort_index())
    bootstrap = MovingBlockBootstrap(n, daily_loss.to_numpy(), seed = 42)

    return bootstrap.conf_int(
        np.mean,
        reps = 2000,
        method = "percentile",
        size = 0.95
    )


In [ ]:
[ci(7), ci(14)]


In [ ]:
# Define the same demand groups for both models using 2024 targets only.
ranked_zones = df_2024.groupby("PULocationID")["trip_count"].mean().sort_values(ascending = False)
rank_zones = ranked_zones.index.tolist()
group_size = len(rank_zones) // 3
high_demand = rank_zones[:group_size]
mid_demand = rank_zones[group_size:2 * group_size]
low_demand = rank_zones[2 * group_size:]

def group_saved_predictions(evaluation, prediction_column):
    # The prediction file was exported in the original 2025 row order.
    keys = ["tpep_pickup_datetime", "PULocationID"]
    pd.testing.assert_frame_equal(evaluation[keys], df_2025[keys])
    assert evaluation.index.equals(pred.index)
    pd.testing.assert_series_equal(evaluation["trip_count"], pred["real"], check_names = False)
    feature_groups = []
    target_prediction_pairs = []
    for zones in [high_demand, mid_demand, low_demand]:
        mask = evaluation["PULocationID"].isin(zones)
        feature_groups.append(evaluation.loc[mask].drop(columns = ["trip_count", "tpep_pickup_datetime"]))
        target_prediction_pairs.extend([
            evaluation.loc[mask, "trip_count"],
            pred.loc[mask, prediction_column],
        ])
    return tuple(feature_groups + target_prediction_pairs)

(basic_high, basic_mid, basic_low, y_high_basic, y_pred_high_basic, y_mid_basic,
 y__pred_mid_basic, y_low_basic, y_pred_low_basic) = group_saved_predictions(df_2025, "pred_basic")


In [ ]:
# Reuse exactly the same zone groups and the saved neighbour-mean predictions.
(mean_high, mean_mid, mean_low, y_high_mean, y_pred_high_mean, y_mid_mean,
 y__pred_mid_mean, y_low_mean, y_pred_low_mean) = group_saved_predictions(df_2025_spatial, "pred_mean")


In [ ]:
zone_metric_rows = []
for label, actual_basic, predicted_basic, actual_mean, predicted_mean in [
    ("High demand", y_high_basic, y_pred_high_basic, y_high_mean, y_pred_high_mean),
    ("Mid demand", y_mid_basic, y__pred_mid_basic, y_mid_mean, y__pred_mid_mean),
    ("Low demand", y_low_basic, y_pred_low_basic, y_low_mean, y_pred_low_mean),
]:
    pd.testing.assert_series_equal(actual_basic, actual_mean)
    assert actual_basic.index.equals(predicted_basic.index)
    assert actual_mean.index.equals(predicted_mean.index)
    zone_metric_rows.append({
        "Demand group": label,
        "MAE — Basic": mean_absolute_error(actual_basic, predicted_basic),
        "MAE — Neighbour mean": mean_absolute_error(actual_mean, predicted_mean),
        "RMSE — Basic": root_mean_squared_error(actual_basic, predicted_basic),
        "RMSE — Neighbour mean": root_mean_squared_error(actual_mean, predicted_mean),
    })

zone_metrics_2025 = pd.DataFrame(zone_metric_rows).set_index("Demand group")
zone_metrics_style = (
    zone_metrics_2025.style.format("{:.4f}")
    .highlight_min(axis = 1, subset = ["MAE — Basic", "MAE — Neighbour mean"], color = "#dcfce7")
    .highlight_min(axis = 1, subset = ["RMSE — Basic", "RMSE — Neighbour mean"], color = "#dcfce7")
    .set_caption("2025 test performance by zone demand — lower is better")
)
zone_metrics_style


In [ ]:
# Paired comparison: positive loss difference favours the neighbour-mean model.
# Average across all zones and hours within each day before resampling days.
assert pred.index.equals(df_2025.index)
pd.testing.assert_series_equal(pred["real"], df_2025["trip_count"], check_names = False)
bootstrap_comparison = pd.DataFrame({
    "date": df_2025["tpep_pickup_datetime"].dt.normalize(),
    "zone": df_2025["PULocationID"],
    "loss_diff": (pred["pred_basic"] - pred["real"])**2
                 - (pred["pred_mean"] - pred["real"])**2,
}, index = pred.index)

bootstrap_group_rows = []
daily_loss_by_group = {}
for label, zones in [("High demand", high_demand), ("Mid demand", mid_demand), ("Low demand", low_demand)]:
    daily_loss = (
        bootstrap_comparison.loc[bootstrap_comparison["zone"].isin(zones)]
        .groupby("date")["loss_diff"].mean().sort_index()
    )
    assert daily_loss.index.equals(pd.date_range("2025-01-01", "2025-12-31", freq = "D"))
    assert daily_loss.notna().all()
    daily_loss_by_group[label] = daily_loss
    for block_days in [7, 14, 28]:
        bs = MovingBlockBootstrap(block_days, daily_loss.to_numpy(), seed = 42)
        lower_ci, upper_ci = bs.conf_int(
            np.mean, reps = 5000, method = "percentile", size = 0.95
        ).ravel()
        conclusion = (
            "Neighbour mean better" if lower_ci > 0 else
            "Basic better" if upper_ci < 0 else
            "Interval includes zero"
        )
        bootstrap_group_rows.append({
            "Demand group": label,
            "Block (days)": block_days,
            "Mean MSE difference": daily_loss.mean(),
            "95% CI lower": lower_ci,
            "95% CI upper": upper_ci,
            "Conclusion": conclusion,
        })

bootstrap_group_results = pd.DataFrame(bootstrap_group_rows).set_index(["Demand group", "Block (days)"])
print("2025 moving-block bootstrap — basic squared loss minus neighbour-mean squared loss")
print("365 daily observations per group; 5,000 bootstrap replications; 95% percentile intervals.")
print(bootstrap_group_results.to_string(float_format = lambda value: f"{value:.4f}"))
print("\nThese are separate, unadjusted intervals for fixed fitted models; they do not include retraining uncertainty.")


## Zone-level prediction errors


In [ ]:
zone_hourly_pu = df_2024.groupby(["PULocationID"])["trip_count"].mean()

len(zone_hourly_pu)


In [ ]:
errors = pred.assign(
    error_basic = (pred["pred_basic"] - pred["real"])**2,
    error_spatial = (pred["pred_mean"] - pred["real"])**2
)

zone_hour_rmse = (
    errors.groupby(["zone"])[["error_basic", "error_spatial"]]
    .mean()
    .apply(np.sqrt)
    .rename(columns = {
        "error_basic": "rmse_basic",
        "error_spatial": "rmse_spatial"
    })
    .reset_index()
)

zone_rmse_improv = [zone_hour_rmse.rmse_basic.loc[i] - zone_hour_rmse.rmse_spatial.loc[i] for i in range(len(zone_hour_rmse))]


In [ ]:
plt.scatter(zone_hourly_pu, zone_hour_rmse.rmse_basic, color = "blue")
plt.ylabel("RMSE")
plt.xlabel("Zone Hourly means")
plt.show()


In [ ]:
plt.scatter(zone_hourly_pu, zone_hour_rmse.rmse_spatial, color = "orange")
plt.ylabel("RMSE")
plt.xlabel("Zone Hourly means")
plt.show()


In [ ]:
plt.scatter(zone_hourly_pu, zone_rmse_improv)
plt.plot([x for x in range(263)], [0 for _ in range(263)], color = "red")
plt.xlabel("Zone Hourly means")
plt.ylabel("RMSE difference (Basic - Spatial)")
plt.show()


In [ ]:
top_rmse_improv = sorted(zone_rmse_improv, reverse = True)
total = np.sum(top_rmse_improv)
top5_part = np.sum(top_rmse_improv[:5])/total
top10_part = np.sum(top_rmse_improv[:10])/total

top5_part, top10_part


In [ ]:
R_zone = [100*zone_rmse_improv[i]/zone_hour_rmse.rmse_basic[i] for i in range(len(zone_rmse_improv))]


In [ ]:
plt.scatter(zone_hourly_pu, R_zone)
plt.show()


In [ ]:
# Inspect the two largest relative RMSE deteriorations.
outliers = zone_hour_rmse.assign(
    relative_rmse_change=R_zone
).nsmallest(2, "relative_rmse_change")["zone"].tolist()
outliers


### Neighbours of outlier zones 2 and 199

Use the same boundary-intersection definition as the adjacency matrix: shared line segments or isolated touching points count as neighbours. Demand statistics below are 2024 hourly averages, including zero-pickup hours. Airport neighbours may have very different demand levels from the target zone; this is a possible explanation to investigate, not proof of why model performance differs.


In [ ]:
from IPython.display import display
from pathlib import Path
import geopandas as gpd

# Load zone geometries here so this section runs independently of notebook 9.
df_zones = gpd.read_file(
    project_root / "taxi_zones" / "taxi_zones.shp"
)

zone_average_2024 = df_2024.groupby("PULocationID")["trip_count"].mean()
zone_info_rows = []
neighbor_info_rows = []
for target_zone in [2, 199]:
    target_row = df_zones.loc[df_zones["LocationID"] == target_zone].iloc[0]
    intersections = df_zones.boundary.intersection(target_row.geometry.boundary)
    neighbor_mask = (~intersections.is_empty) & (df_zones["LocationID"] != target_zone)
    target_neighbors = df_zones.loc[neighbor_mask].sort_values("LocationID")
    zone_info_rows.append({
        "Zone ID": target_zone,
        "Zone name": target_row["zone"],
        "Borough": target_row["borough"],
        "Neighbours": len(target_neighbors),
        "Mean hourly pickups (2024)": zone_average_2024.loc[target_zone],
    })
    for index, neighbor in target_neighbors.iterrows():
        shared = intersections.loc[index]
        neighbor_info_rows.append({
            "Target zone": target_zone,
            "Neighbour ID": int(neighbor["LocationID"]),
            "Neighbour name": neighbor["zone"],
            "Borough": neighbor["borough"],
            "Contact": "Shared boundary segment" if shared.length > 0 else "Point contact only",
            "Mean hourly pickups (2024)": zone_average_2024.loc[neighbor["LocationID"]],
        })

outlier_zone_info = pd.DataFrame(zone_info_rows).set_index("Zone ID")
outlier_neighbor_info = pd.DataFrame(neighbor_info_rows).set_index(["Target zone", "Neighbour ID"])
display(outlier_zone_info.round(4))
display(outlier_neighbor_info.round(4))


### Historical demand contrast and 2025 RMSE improvement

For zone $i$, define contrast as the **unweighted mean of its neighbours' 2024 hourly demand means minus its own 2024 hourly demand mean**. Use the same boundary-contact neighbourhood as the spatial feature. Positive contrast means busier neighbours. Improvement is **2025 basic RMSE minus neighbour-model RMSE**; positive improvement favours the neighbour model. Zones without neighbours have undefined contrast and are excluded from the scatter plot.


In [ ]:
historical_zone_means = df_2024.groupby("PULocationID")["trip_count"].mean()
contrast_rows = []
for zone_id, zone_row in df_zones.set_index("LocationID").sort_index().iterrows():
    shared_boundaries = df_zones.boundary.intersection(zone_row.geometry.boundary)
    neighbor_ids = df_zones.loc[
        (~shared_boundaries.is_empty) & (df_zones["LocationID"] != zone_id), "LocationID"
    ]
    own_mean = historical_zone_means.loc[zone_id]
    neighbor_mean = historical_zone_means.reindex(neighbor_ids).mean() if len(neighbor_ids) else np.nan
    contrast_rows.append({
        "zone": zone_id,
        "zone_name": zone_row["zone"],
        "neighbor_count": len(neighbor_ids),
        "zone_mean_2024": own_mean,
        "neighbor_mean_2024": neighbor_mean,
        "demand_contrast_2024": neighbor_mean - own_mean,
    })

demand_contrast_2024 = pd.DataFrame(contrast_rows).set_index("zone")
# Join by zone ID rather than relying on the order of separate arrays.
contrast_vs_improvement = demand_contrast_2024.join(
    zone_hour_rmse.set_index("zone")[["rmse_basic", "rmse_spatial"]], validate = "one_to_one"
)
contrast_vs_improvement["rmse_improvement_2025"] = (
    contrast_vs_improvement["rmse_basic"] - contrast_vs_improvement["rmse_spatial"]
)
assert len(contrast_vs_improvement) == 263
assert contrast_vs_improvement[["rmse_basic", "rmse_spatial"]].notna().all().all()
print("Demand contrast uses 2024 only; RMSE improvement uses 2025 predictions.")
print(f"{contrast_vs_improvement['demand_contrast_2024'].notna().sum()} zones have a defined contrast; "
      f"{contrast_vs_improvement['demand_contrast_2024'].isna().sum()} isolated zones are excluded from the plot.")


In [ ]:
contrast_plot_data = contrast_vs_improvement.dropna(subset = ["demand_contrast_2024"])
fig, ax = plt.subplots(figsize = (10, 6))
ax.scatter(
    contrast_plot_data["demand_contrast_2024"],
    contrast_plot_data["rmse_improvement_2025"],
    alpha = 0.7, s = 30, color = "tab:blue",
)
ax.axhline(0, color = "grey", linestyle = "--", linewidth = 1)
ax.axvline(0, color = "grey", linestyle = ":", linewidth = 1)
for zone_id, offset in [(2, (8, 15)), (199, (8, -20))]:
    row = contrast_plot_data.loc[zone_id]
    ax.scatter(row["demand_contrast_2024"], row["rmse_improvement_2025"], color = "tab:red", s = 45, zorder = 3)
    ax.annotate(f"{zone_id}: {row['zone_name']}",
                (row["demand_contrast_2024"], row["rmse_improvement_2025"]),
                xytext = offset, textcoords = "offset points", fontsize = 9,
                arrowprops = {"arrowstyle": "-", "color": "tab:red"})
ax.set_xlabel("2024 demand contrast: neighbours' mean − zone mean (pickups/hour)")
ax.set_ylabel("2025 RMSE improvement: basic − neighbour mean")
ax.set_title("Historical neighbour-demand contrast vs prediction improvement")
ax.grid(alpha = 0.15)
fig.tight_layout()
contrast_project_dir = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
contrast_plot_path = contrast_project_dir / "reports" / "figures" / "demand_contrast_2024_vs_rmse_improvement_2025.png"
contrast_plot_path.parent.mkdir(parents = True, exist_ok = True)
fig.savefig(contrast_plot_path, dpi = 150, bbox_inches = "tight")
plt.show()


### Demand contrast by 2024 demand group

Reuse `high_demand`, `mid_demand`, and `low_demand`, ranked on 2024 average hourly demand. Neighbour means still use **all adjacent zones**, including neighbours in other demand groups. Only the target zones are grouped. Each panel has its own axis scales so patterns within the group remain visible. Zones without neighbours are omitted.


In [ ]:
contrast_demand_groups = {
    "High demand": high_demand,
    "Mid demand": mid_demand,
    "Low demand": low_demand,
}
contrast_by_group = {}
contrast_group_counts = []
for label, zone_ids in contrast_demand_groups.items():
    group_rows = contrast_vs_improvement.loc[contrast_vs_improvement.index.isin(zone_ids)].copy()
    plot_rows = group_rows.dropna(subset = ["demand_contrast_2024", "rmse_improvement_2025"])
    contrast_by_group[label] = plot_rows
    contrast_group_counts.append({
        "Demand group": label,
        "Total zones": len(group_rows),
        "Plotted zones": len(plot_rows),
        "No neighbours": int(group_rows["neighbor_count"].eq(0).sum()),
    })
contrast_group_summary = pd.DataFrame(contrast_group_counts).set_index("Demand group")
assert contrast_group_summary["Total zones"].sum() == len(contrast_vs_improvement)
assert sum(len(rows) for rows in contrast_by_group.values()) == contrast_vs_improvement["demand_contrast_2024"].notna().sum()
print(contrast_group_summary.to_string())


In [ ]:
fig_group_contrast, group_axes = plt.subplots(1, 3, figsize = (17, 5.5))
for ax, (label, rows), color in zip(group_axes, contrast_by_group.items(), ["tab:blue", "tab:orange", "tab:green"]):
    ax.scatter(rows["demand_contrast_2024"], rows["rmse_improvement_2025"], color = color, alpha = 0.75, s = 30)
    ax.axhline(0, color = "grey", linestyle = "--", linewidth = 1)
    ax.axvline(0, color = "grey", linestyle = ":", linewidth = 1)
    ax.set_title(f"{label} ({len(rows)} zones)")
    ax.set_xlabel("2024 demand contrast\n(neighbours' mean − zone mean, pickups/hour)")
    ax.set_ylabel("2025 RMSE improvement\n(basic − neighbour mean)")
    ax.grid(alpha = 0.15)
    for zone_id, offset in [(2, (-65, 24)), (199, (-110, 20))]:
        if zone_id in rows.index:
            row = rows.loc[zone_id]
            ax.scatter(row["demand_contrast_2024"], row["rmse_improvement_2025"], color = "tab:red", s = 45, zorder = 3)
            ax.annotate(f"{zone_id}: {row['zone_name']}",
                        (row["demand_contrast_2024"], row["rmse_improvement_2025"]),
                        xytext = offset, textcoords = "offset points", fontsize = 8,
                        arrowprops = {"arrowstyle": "-", "color": "tab:red"})
fig_group_contrast.suptitle("Demand contrast vs RMSE improvement by 2024 demand group — independent panel scales")
fig_group_contrast.tight_layout(rect = [0, 0, 1, 0.94])
group_contrast_project_dir = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
group_contrast_plot_path = group_contrast_project_dir / "reports" / "figures" / "demand_contrast_vs_rmse_improvement_by_group.png"
group_contrast_plot_path.parent.mkdir(parents = True, exist_ok = True)
fig_group_contrast.savefig(group_contrast_plot_path, dpi = 150, bbox_inches = "tight")
plt.show()


In [ ]:
# Select low-demand zones by zone ID, then remove the requested row indices.
low_rmse_spatial = zone_hour_rmse.loc[
    zone_hour_rmse["zone"].isin(low_demand), "rmse_spatial"
].drop(index = [1, 198, 202, 52, 123, 193, 218], errors = "ignore")

low_rmse_basic = zone_hour_rmse.loc[
    zone_hour_rmse["zone"].isin(low_demand), "rmse_basic"
].drop(index = [1, 198, 202, 52, 123, 193, 218], errors = "ignore")

np.mean(low_rmse_basic) - np.mean(low_rmse_spatial)


### Zone categories based only on 2024 historical demand

1. Zone mean < 1 pickup/hour and neighbours' mean >= 10 pickups/hour.
2. Zone mean < 1 pickup/hour and neighbours' mean < 10 pickups/hour.
3. Zone mean >= 1 pickup/hour, regardless of neighbours.

Neighbour demand is the unweighted average of adjacent zones' historical means. Low-demand zones without neighbours cannot be assigned to category 1 or 2 and are reported separately rather than treating an undefined mean as zero. These groups are descriptive and do not use 2025 errors; the thresholds were proposed after examining the test results.


In [ ]:
# Reuse historical statistics; no 2025 targets or prediction errors enter this grouping.
zone_categories_2024 = demand_contrast_2024[[
    "zone_name", "neighbor_count", "zone_mean_2024", "neighbor_mean_2024"
]].copy()
zone_categories_2024["category"] = pd.Series(pd.NA, index = zone_categories_2024.index, dtype = "string")
own_low = zone_categories_2024["zone_mean_2024"] < 1
neighbors_busy = zone_categories_2024["neighbor_mean_2024"] >= 10
neighbors_quiet = zone_categories_2024["neighbor_mean_2024"] < 10
own_higher = zone_categories_2024["zone_mean_2024"] >= 1

zone_categories_2024.loc[own_low & neighbors_busy, "category"] = "1: Low demand, busy neighbours"
zone_categories_2024.loc[own_low & neighbors_quiet, "category"] = "2: Low demand, quiet neighbours"
zone_categories_2024.loc[own_higher, "category"] = "3: Demand >= 1 pickup/hour"

zones_low_busy_neighbors = zone_categories_2024.index[own_low & neighbors_busy].tolist()
zones_low_quiet_neighbors = zone_categories_2024.index[own_low & neighbors_quiet].tolist()
zones_higher_demand = zone_categories_2024.index[own_higher].tolist()
zones_without_neighbor_reference = zone_categories_2024.index[zone_categories_2024["category"].isna()].tolist()

zone_categories_summary = pd.DataFrame({
    "Category": ["1: Low demand, busy neighbours", "2: Low demand, quiet neighbours", "3: Demand >= 1 pickup/hour", "Unclassified: low demand, no neighbours"],
    "Zones": [len(zones_low_busy_neighbors), len(zones_low_quiet_neighbors), len(zones_higher_demand), len(zones_without_neighbor_reference)],
}).set_index("Category")

print(zone_categories_summary.to_string())


In [ ]:
low_demand_busy_neighbours_rmse = [zone_rmse_improv[i-1] for i in zones_low_busy_neighbors]
low_demand_quiet_neighbours_rmse = [zone_rmse_improv[i-1] for i in zones_low_quiet_neighbors]
zones_higher_demand_rmse = [zone_rmse_improv[i-1] for i in zones_higher_demand]
no_neighbors_rmse = [zone_rmse_improv[i-1] for i in zones_without_neighbor_reference]


In [ ]:
# Descriptive mean of per-zone RMSE gains; not a pooled group RMSE difference.
category_mean_improvement = pd.DataFrame([
    {"Category": "Low demand, busy neighbours", "Zones": len(low_demand_busy_neighbours_rmse),
     "Mean zone RMSE improvement": np.mean(low_demand_busy_neighbours_rmse)},
    {"Category": "Low demand, quiet neighbours", "Zones": len(low_demand_quiet_neighbours_rmse),
     "Mean zone RMSE improvement": np.mean(low_demand_quiet_neighbours_rmse)},
    {"Category": "Demand >= 1 pickup/hour", "Zones": len(zones_higher_demand_rmse),
     "Mean zone RMSE improvement": np.mean(zones_higher_demand_rmse)},
    {"Category": "Low demand, no neighbours", "Zones": len(no_neighbors_rmse),
     "Mean zone RMSE improvement": np.mean(no_neighbors_rmse)},
]).set_index("Category")

category_mean_style = (
    category_mean_improvement.style
    .format({"Zones": "{:d}", "Mean zone RMSE improvement": "{:+.6f}"})
    .set_caption("2025 mean per-zone RMSE improvement by 2024 category — basic minus spatial; positive favours spatial")
)
category_mean_style
